# Fase 4 — Scoring Metagenomico HPC: aceleracion GPU con CUDA (Google Colab)

**Nivel 3.** Random Search sobre el simplex `W=(W1,W2,W3)` que maximiza `AUC(y, Score(W))`,
**equivalente a las Fases 1-3** (Python secuencial/multicore, C+OpenMP, C+MPI): mismo *valor*
de AUC (`|dAUC| < 1e-4`, RF-04) y mismo esquema de benchmark de 10 columnas (DEC-13).

Un hilo GPU evalua un candidato `W_k`; el RNG SplitMix64, `P=W*[T,S,F]`, `Score=A*P` y el AUC
por conteo de pares con empates a 0.5 son copias del computo ya validado en `scoring_openmp.c`.
Cada hilo reconstruye `W_k = sample_dirichlet(seed + k)` desde su indice global (DEC-12), por lo
que el resultado es determinista y no se transfiere ningun `W_pool`.

**Entorno:** ejecutar en un runtime **GPU de Google Colab** (tipicamente NVIDIA T4). No requiere
GPU local (RIESGO-01). `scoring_kernel.cu` y `scoring_pycuda.py` son fuentes derivadas (DEC-09).

**Parametros (DEC-03/04/05/10):** `N_ITEMS=50`, `K=100000`, `SEED=42`, `BLOCK_SIZE=256`, `N_SAMPLES=10`.

> **Placeholders / valores que dependen de la corrida en Colab:** `GPU_NAME`, `best_auc`,
> `t_search` (`time_seconds`), `speedup_vs_python` y la fila CSV final **se calculan al ejecutar
> en Colab**. En el repositorio el notebook se versiona **sin salidas** (no se ejecuta en local).
> Tras correr en Colab: (1) anotar el modelo de GPU; (2) **transcribir** la fila `CUDA` impresa por
> la celda 11 a `code/results/benchmark.csv` (append-only, sin tocar filas previas ni el esquema).


In [ ]:
# (0) Parametros globales (DEC-03/DEC-04/DEC-05/DEC-10/DEC-13)
N_ITEMS = 50          # tamano del problema -> data/n_50/
K = 100_000           # candidatos por busqueda (DEC-04)
SEED = 42             # semilla fija (DEC-03)
BLOCK_SIZE = 256      # hilos por bloque (DEC-05)
N_SAMPLES = 10        # 5 sanas (y=0) + 5 enfermas (y=1)
MASK64 = (1 << 64) - 1

# Baseline para speedup_vs_python: fila 'Python secuencial' n_50/K=100000 de
# code/results/benchmark.csv (time_seconds = 96.30376639, DEC-13).
T_PYTHON_SEQ = 96.30376639

# AUC de referencia de Python secuencial (n_50/K=100k, seed=42): dataset separable -> 1.0
AUC_PYTHON_SEQ = 1.0

print(f"N_ITEMS={N_ITEMS} K={K} SEED={SEED} BLOCK_SIZE={BLOCK_SIZE} N_SAMPLES={N_SAMPLES}")


## 1. GPU — verificar runtime y registrar el modelo (RIESGO-06)

In [ ]:
# (1) Confirmar runtime GPU y registrar el modelo (RNF-03 / RIESGO-06)
get_ipython().system('nvidia-smi')
import subprocess
try:
    GPU_NAME = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
    ).strip().splitlines()[0]
except Exception as exc:   # sin GPU (entorno local): no ejecutable, documentar
    GPU_NAME = f"DESCONOCIDA ({exc})"
print("Modelo de GPU:", GPU_NAME)


## 2. Setup PyCUDA (instalar solo si falta; sin nuevas dependencias)

In [ ]:
# (2) Importar PyCUDA; instalar SOLO si el import falla (no se anaden dependencias al proyecto)
try:
    import pycuda.autoinit
    import pycuda.driver as cuda
    from pycuda.compiler import SourceModule
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycuda"], check=True)
    import pycuda.autoinit
    import pycuda.driver as cuda
    from pycuda.compiler import SourceModule

import numpy as np
import time
from math import log
print("PyCUDA OK | device:", cuda.Context.get_device().name())


## Funciones de referencia en host

Mirror **exacto** del RNG del kernel (SplitMix64) y de las ecuaciones 1-4 (copias de
`scoring_openmp.c` / `common.py`). Sirven para reconstruir `best_W`, el self-test y la validacion.


In [ ]:
# Mirror Python del SplitMix64->Dirichlet __device__ (mismo stream de bits que el kernel).
def sample_dirichlet_host(seed_k):
    state = seed_k & MASK64
    e = []
    for _ in range(3):
        state = (state + 0x9E3779B97F4A7C15) & MASK64
        z = state
        z = ((z ^ (z >> 30)) * 0xBF58476D1CE4E5B9) & MASK64
        z = ((z ^ (z >> 27)) * 0x94D049BB133111EB) & MASK64
        z = z ^ (z >> 31)
        u = (z >> 11) * (1.0 / 9007199254740992.0) + (1.0 / 18014398509481984.0)
        e.append(-log(u))
    s = sum(e)
    return np.array([v / s for v in e], dtype=np.float64)

# P = W1*T + W2*S + W3*F ; Score = A @ P (ecs. 1-2).
def host_score(A, profiles, W):
    T, S, F = profiles[:, 0], profiles[:, 1], profiles[:, 2]
    P = W[0] * T + W[1] * S + W[2] * F
    return A @ P

# AUC por conteo de pares con empates a 0.5 (ec. 3, igual al kernel y a sklearn).
def host_auc(score, labels):
    pos = [i for i in range(len(labels)) if labels[i] == 1]
    neg = [i for i in range(len(labels)) if labels[i] == 0]
    if not pos or not neg:
        return 0.5
    c = 0.0
    for i in pos:
        for j in neg:
            if score[i] > score[j]:
                c += 1.0
            elif score[i] == score[j]:
                c += 0.5
    return c / (len(pos) * len(neg))

# Consistencia de scoring (ec. 4): TP/5 + TN/5 con theta = mediana(score).
def host_consistency(score, labels):
    theta = float(np.median(score))
    tp = int(np.sum((score > theta) & (labels == 1)))
    tn = int(np.sum((score <= theta) & (labels == 0)))
    return tp / 5 + tn / 5


## 3. Carga y validacion de `.npy` desde `code/data/n_50/` (DEC-10, solo lectura)

Sube los tres `.npy` al runtime de Colab por **una** via (documentar la elegida):
**(A)** `files.upload()`, **(B)** `git clone` del repo y apuntar `DATA_DIR` a `code/data/n_50/`,
o **(C)** montar Google Drive. La carga ocurre **fuera del cronometro**.


In [ ]:
# (3) Cargar y validar el dataset (espejo de common.validate_dataset)
from pathlib import Path

DATA_DIR = Path("data/n_50")    # <-- AJUSTAR a la ruta en Colab (p.ej. tras git clone del repo)

# Opcion A (subida manual): descomentar
# from google.colab import files
# files.upload()           # subir matrix_A.npy, profiles.npy, labels.npy
# DATA_DIR = Path(".")

def validate_dataset(A, profiles, y, n_items):
    assert A.shape == (N_SAMPLES, n_items) and A.dtype == np.float32, (A.shape, A.dtype)
    assert profiles.shape == (n_items, 3) and profiles.dtype == np.float32, profiles.shape
    assert y.shape == (N_SAMPLES,) and y.dtype == np.int32, (y.shape, y.dtype)
    assert np.array_equal(y, np.array([0]*5 + [1]*5, dtype=np.int32)), y
    T, S, F = profiles[:, 0], profiles[:, 1], profiles[:, 2]
    assert np.all((T >= 0) & (T <= 1)) and np.all((S >= 0) & (S <= 1)), "T/S fuera de [0,1]"
    assert np.all(np.isin(F, [0, 1, 2])), "F fuera de {0,1,2}"

A = np.ascontiguousarray(np.load(DATA_DIR / "matrix_A.npy"), dtype=np.float32)
profiles = np.ascontiguousarray(np.load(DATA_DIR / "profiles.npy"), dtype=np.float32)
y = np.ascontiguousarray(np.load(DATA_DIR / "labels.npy"), dtype=np.int32)
validate_dataset(A, profiles, y, N_ITEMS)
print("Datos OK:", A.shape, profiles.shape, y.shape)


## 4. Kernel CUDA (`%%writefile scoring_kernel.cu`)

Materializa la fuente derivada (DEC-09). Contiene **solo** funciones `__device__` + `__global__`
(sin `main`): RNG SplitMix64, `P=W*[T,S,F]`, `Score=A*P` con `A`/`profiles` en `__shared__`, y AUC
por conteo de pares con empates a 0.5. `BLOCK_SIZE=256` (DEC-05); `< 400 LOC`.


In [ ]:
%%writefile scoring_kernel.cu
/*
 * Nivel 3 — CUDA: kernel de Random Search sobre el simplex W=(W1,W2,W3) que
 * maximiza AUC(y, Score(W)). Un hilo GPU evalua un candidato W_k (DEC-05/DEC-15).
 *
 * Equivalente por construccion a python/sequential.py, scoring_openmp.c y
 * scoring_mpi.c: el RNG SplitMix64, P=W*[T,S,F], Score=A*P y el AUC por conteo de
 * pares con empates a 0.5 son copias literales de scoring_openmp.c (Fase 2). Cada
 * hilo reconstruye W_k = sample_dirichlet(seed + k) desde su indice global k
 * (DEC-12) => mismo valor de AUC, determinista y sin transferir W_pool.
 *
 * Fuente derivada (DEC-09): este .cu se materializa con %%writefile dentro de
 * CUDA/scoring_cuda.ipynb y se compila con pycuda.compiler.SourceModule(-O2).
 * Contiene SOLO funciones __device__ + __global__ (sin main), por lo que para un
 * chequeo de sintaxis se compila con `nvcc -O2 -c scoring_kernel.cu` (solo objeto).
 *
 * Restricciones: BLOCK_SIZE=256 (DEC-05), seed=42 (DEC-03), < 400 LOC.
 */
#include <math.h>

#define N_SAMPLES 10   /* 5 sanas (y=0) + 5 enfermas (y=1) */
#define BLOCK_SIZE 256 /* DEC-05: multiplo de 32 (warp) */
#define MAX_ITEMS 256  /* cota de n_items para arrays locales/__shared__ (sin VLAs) */

/* ----------------------------------------------------------------------------
 * RNG: SplitMix64 -> Dirichlet(1,1,1) (DEC-12) — copia __device__ de scoring_openmp.c.
 * Aritmetica de 64 bits sin signo: envuelve mod 2^64 igual que el `uint64_t` de C,
 * por lo que el stream de W_k es identico al de las Fases 2/3.
 * -------------------------------------------------------------------------- */
__device__ unsigned long long splitmix64_next(unsigned long long *state) {
    unsigned long long z = (*state += 0x9E3779B97F4A7C15ULL);
    z = (z ^ (z >> 30)) * 0xBF58476D1CE4E5B9ULL;
    z = (z ^ (z >> 27)) * 0x94D049BB133111EBULL;
    return z ^ (z >> 31);
}

/* Uniforme en (0, 1]: evita log(0) en la transformacion exponencial. */
__device__ double splitmix64_uniform01(unsigned long long *state) {
    unsigned long long v = splitmix64_next(state);
    return (double)(v >> 11) * (1.0 / 9007199254740992.0) + (1.0 / 18014398509481984.0);
}

/* Dirichlet(1,1,1) = 3 exponenciales(1) = -log(U) normalizadas.
 * W_k depende solo de seed_k (DEC-12): sin estado compartido entre hilos. */
__device__ void sample_dirichlet(double W[3], unsigned long long seed_k) {
    unsigned long long state = seed_k;
    double e[3], sum = 0.0;
    for (int i = 0; i < 3; i++) {
        e[i] = -log(splitmix64_uniform01(&state));
        sum += e[i];
    }
    for (int i = 0; i < 3; i++) W[i] = e[i] / sum;
}

/* ----------------------------------------------------------------------------
 * AUC por conteo de pares con empates acreditados a 0.5 (ec. 3, igual a
 * sklearn.roc_auc_score y a compute_auc de scoring_openmp.c, RIESGO-03).
 * Acumula en `double`. `labels` y `score` tienen longitud n (= N_SAMPLES).
 * -------------------------------------------------------------------------- */
__device__ double compute_auc_dev(const double *score, const int *labels, int n) {
    int pos = 0, neg = 0;
    for (int i = 0; i < n; i++) {
        if (labels[i] == 1) pos++;
        else neg++;
    }
    if (pos == 0 || neg == 0) return 0.5;

    double concordant = 0.0;
    for (int i = 0; i < n; i++) {
        if (labels[i] != 1) continue;
        for (int j = 0; j < n; j++) {
            if (labels[j] != 0) continue;
            if (score[i] > score[j]) concordant += 1.0;
            else if (score[i] == score[j]) concordant += 0.5;
        }
    }
    return concordant / ((double)pos * (double)neg);
}

/* ----------------------------------------------------------------------------
 * Kernel principal: un hilo = un candidato. `A` (N_SAMPLES x n_items, C-order) y
 * `profiles` (n_items x 3 = T,S,F) se cachean en memoria compartida por bloque
 * (carga cooperativa + __syncthreads()); los 256 hilos del bloque los reutilizan.
 * P (vector dim n_items) vive en un array local dimensionado por MAX_ITEMS (sin VLAs).
 * -------------------------------------------------------------------------- */
__global__ void scoring_kernel(const float *A,        /* N_SAMPLES x n_items */
                               const float *profiles,  /* n_items x 3 (T,S,F)  */
                               const int *labels,      /* N_SAMPLES            */
                               float *auc_out,         /* K resultados         */
                               int n_items, int K, unsigned long long seed) {
    __shared__ float sA[N_SAMPLES * MAX_ITEMS]; /* ~10 KB para n_items=50 */
    __shared__ float sProf[MAX_ITEMS * 3];      /* ~0.6 KB                */
    __shared__ int sY[N_SAMPLES];

    /* Carga cooperativa: TODOS los hilos del bloque participan antes del return. */
    for (int idx = threadIdx.x; idx < N_SAMPLES * n_items; idx += blockDim.x)
        sA[idx] = A[idx];
    for (int idx = threadIdx.x; idx < n_items * 3; idx += blockDim.x)
        sProf[idx] = profiles[idx];
    if (threadIdx.x < N_SAMPLES) sY[threadIdx.x] = labels[threadIdx.x];
    __syncthreads();

    int k = blockIdx.x * blockDim.x + threadIdx.x;
    if (k >= K) return; /* despues de la barrera: no hay deadlock */

    /* W_k ~ Dirichlet(1,1,1) reconstruido desde el indice global k (DEC-12). */
    double W[3];
    sample_dirichlet(W, seed + (unsigned long long)k);

    /* P_i = W0*T_i + W1*S_i + W2*F_i (ec. 1). */
    double P[MAX_ITEMS];
    for (int i = 0; i < n_items; i++) {
        double T = sProf[i * 3 + 0], S = sProf[i * 3 + 1], F = sProf[i * 3 + 2];
        P[i] = W[0] * T + W[1] * S + W[2] * F;
    }

    /* Score_j = sum_i A[j,i] * P_i (ec. 2). */
    double score[N_SAMPLES];
    for (int j = 0; j < N_SAMPLES; j++) {
        double s = 0.0;
        for (int i = 0; i < n_items; i++) s += (double)sA[j * n_items + i] * P[i];
        score[j] = s;
    }

    auc_out[k] = (float)compute_auc_dev(score, sY, N_SAMPLES);
}


## 5. Compilacion con `SourceModule(-O2)` (+ `nvcc -c` opcional como sanity)

In [ ]:
# (5) Compilar el kernel con SourceModule(-O2) (JIT) para orquestarlo en el mismo flujo.
KERNEL_SRC = Path("scoring_kernel.cu").read_text()
module = SourceModule(KERNEL_SRC, options=["-O2"])
scoring_kernel = module.get_function("scoring_kernel")
print("Kernel compilado con SourceModule(-O2)")

# (5-opcional) Sanity de sintaxis con nvcc. El .cu NO tiene main (solo __device__/__global__),
# por eso se compila a OBJETO con -c (no se enlaza un ejecutable).
get_ipython().system('nvcc -O2 -c scoring_kernel.cu -o /tmp/scoring_kernel.o && echo "nvcc -c OK"')


## 6-8. Orquestacion PyCUDA + timing (excluye carga de datos)

H2D **una sola vez** de `A`/`profiles`/`labels`; lanzamiento `grid=ceil(K/256)`, `block=256`;
D2H de `auc_out`. El cronometro `perf_counter` envuelve [lanzamiento -> sync -> D2H -> argmax]
(= tiempo de **busqueda de W***, el `time_seconds` del CSV); `cudaEvent_t` mide el kernel puro.
Se **excluye** la carga `.npy` y la H2D unica (DEC-15 / rules.md, Benchmark).


In [ ]:
# (6) Orquestacion + (8) timing. CUDA_CHECK: PyCUDA ya lanza excepcion ante errores CUDA;
# sincronizar el contexto fuerza a aflorar errores asincronos del kernel.
def CUDA_CHECK(msg=""):
    cuda.Context.synchronize()

def gpu_random_search(A, profiles, y, n_items, K, seed, block_size=BLOCK_SIZE):
    # H2D UNA sola vez (fuera del cronometro: analoga a 'carga de datos').
    dA = cuda.mem_alloc(A.nbytes); cuda.memcpy_htod(dA, A)
    dProf = cuda.mem_alloc(profiles.nbytes); cuda.memcpy_htod(dProf, profiles)
    dY = cuda.mem_alloc(y.nbytes); cuda.memcpy_htod(dY, y)
    auc_out = np.empty(K, dtype=np.float32)
    dAuc = cuda.mem_alloc(auc_out.nbytes)

    grid = ((K + block_size - 1) // block_size, 1)   # ceil(K/256)
    block = (block_size, 1, 1)
    start, end = cuda.Event(), cuda.Event()

    cuda.Context.synchronize()
    t0 = time.perf_counter()                          # (8) perf_counter -> busqueda de W*
    start.record()                                    # (8) cudaEvent -> kernel puro
    scoring_kernel(dA, dProf, dY, dAuc, np.int32(n_items), np.int32(K),
                   np.uint64(seed), block=block, grid=grid)
    end.record(); end.synchronize()
    cuda.memcpy_dtoh(auc_out, dAuc)                   # D2H de auc_out
    k_star = int(np.argmax(auc_out))                  # (7) reduccion en host
    cuda.Context.synchronize()
    t_search = time.perf_counter() - t0
    kernel_ms = start.time_till(end)
    CUDA_CHECK("scoring_kernel")
    return auc_out, k_star, t_search, kernel_ms

auc_out, k_star, t_search, kernel_ms = gpu_random_search(A, profiles, y, N_ITEMS, K, SEED)
print(f"k* = {k_star} | auc_out[k*] = {auc_out[k_star]:.6f}")


## 7. Reduccion: `np.argmax` en host + reconstruccion de `best_W`

In [ ]:
# (7) argmax en host (primer maximo => menor k, igual a Fase 1 y al MAXLOC de Fase 3)
#     + reconstruccion de best_W con el MISMO SplitMix64 (DEC-12/DEC-15).
best_auc = float(auc_out[k_star])
best_W = sample_dirichlet_host((SEED + k_star) & MASK64)

# Verificacion cruzada: recomputar el AUC de best_W en host debe coincidir con la GPU.
best_score = host_score(A, profiles, best_W)
auc_host = host_auc(best_score, y)
print("best_W =", best_W, "| suma =", float(best_W.sum()))
print(f"best_auc (GPU) = {best_auc:.6f} | AUC(best_W) host = {auc_host:.6f}")


## 8. Tiempos (excluyen carga de datos y H2D unica)

In [ ]:
# (8) time_seconds -> busqueda de W* (perf_counter): valor del CSV.
#     kernel_ms     -> kernel puro (cudaEvent): analisis/Amdahl Fase 5.
print(f"time_seconds (busqueda de W*) = {t_search:.6f} s")
print(f"kernel puro (cudaEvent)       = {kernel_ms:.3f} ms")
print(f"candidates_per_second         = {K / t_search:,.0f}")


## 9. Self-test (RIESGO-03)

(a) AUC con empate -> **0.875** exacto (mismo caso que `self_test` en C).
(b) Extremo-a-extremo `n_items=3`/`K=100`: CUDA vs **mirror SplitMix64** en host (mismo RNG) ->
`|dAUC| ~ 0`. Comparar contra `sequential.py` (PCG64) **no** seria apples-to-apples (DEC-15).


In [ ]:
# (9a) Unitario AUC con empate -> 0.875
auc_tie = host_auc(np.array([1, 2, 2, 3], dtype=float), np.array([0, 0, 1, 1]))
SELFTEST_TIE_OK = abs(auc_tie - 0.875) < 1e-9
assert SELFTEST_TIE_OK, auc_tie
print(f"[self-test] AUC empate = {auc_tie} (esperado 0.875) -> OK")

# (9b) Extremo-a-extremo n_items=3 / K=100: CUDA vs mirror SplitMix64 en host (mismo RNG).
rng = np.random.default_rng(123)
n3, K3 = 3, 100
A3 = np.ascontiguousarray(rng.random((N_SAMPLES, n3)), dtype=np.float32)
prof3 = np.empty((n3, 3), dtype=np.float32)
prof3[:, 0] = rng.random(n3)                 # T en [0,1]
prof3[:, 1] = rng.random(n3)                 # S en [0,1]
prof3[:, 2] = rng.integers(0, 3, n3)         # F en {0,1,2}
y3 = np.array([0]*5 + [1]*5, dtype=np.int32)

auc3, k3, _, _ = gpu_random_search(A3, prof3, y3, n3, K3, SEED)
best_auc_cuda3 = float(auc3[k3])

best_host3 = -1.0                            # referencia host con el MISMO SplitMix64
for k in range(K3):
    W = sample_dirichlet_host((SEED + k) & MASK64)
    a = host_auc(host_score(A3, prof3, W), y3)
    if a > best_host3:
        best_host3 = a

SELFTEST_SMALL_OK = abs(best_auc_cuda3 - best_host3) < 1e-4
print(f"[self-test] n=3/K=100: CUDA={best_auc_cuda3:.6f} host={best_host3:.6f} "
      f"|dAUC|={abs(best_auc_cuda3 - best_host3):.2e}")
assert SELFTEST_SMALL_OK
print("[self-test] n=3/K=100 -> OK")


## 10. Validacion de correctitud (RF-04 / RNF-02), caso principal n_50/K=100k

In [ ]:
# (10) Validacion (caso principal n_50/K=100k)
consistency = host_consistency(best_score, y)
print(f"best_auc = {best_auc:.6f} | consistencia = {consistency:.4f}")

assert 0.5 <= best_auc <= 1.0, "AUC fuera de [0.5,1.0]"                       # RNF-02
assert abs(best_auc - AUC_PYTHON_SEQ) < 1e-4, "|AUC_CUDA - AUC_PySeq| >= 1e-4"  # RF-04
assert abs(best_auc - auc_host) < 1e-4, "GPU vs host divergen"                # kernel vs host
assert consistency >= 0.8, "consistencia < 0.8"                              # RNF-02
print("Validacion OK: AUC in [0.5,1], |dAUC|<1e-4 vs PySeq, consistencia>=0.8")


## 11. Fila `CUDA` del benchmark (10 columnas, DEC-13)

Una sola fila (sin barrido P): `workers=1` (1 GPU) => `speedup=1.0`, `efficiency=1.0`;
`speedup_vs_python = T_PYTHON_SEQ / t_search`. La fila se **imprime para transcribirla** (append-only)
a `code/results/benchmark.csv` del repo, **sin tocar filas previas ni el esquema**.


In [ ]:
# (11) Construir y mostrar la fila CSV 'CUDA' (DEC-13)
import csv, io
COLUMNS = ["implementation", "n_items", "k_candidates", "workers", "best_auc", "time_seconds",
           "candidates_per_second", "speedup", "efficiency", "speedup_vs_python"]
row = {
    "implementation": "CUDA",
    "n_items": N_ITEMS,
    "k_candidates": K,
    "workers": 1,
    "best_auc": best_auc,
    "time_seconds": t_search,
    "candidates_per_second": K / t_search,
    "speedup": 1.0,
    "efficiency": 1.0,
    "speedup_vs_python": T_PYTHON_SEQ / t_search,
}
buf = io.StringIO()
csv.DictWriter(buf, fieldnames=COLUMNS).writerow(row)
print("Fila a TRANSCRIBIR (append-only) en code/results/benchmark.csv:\n")
print(",".join(COLUMNS))
print(buf.getvalue().strip())

# Opcional en Colab: guardar la fila para descargarla y transcribirla luego.
with open("benchmark_cuda_row.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=COLUMNS); w.writeheader(); w.writerow(row)
print("\n(Guardada tambien en benchmark_cuda_row.csv)")


## 12. Resumen y criterios de salida (tarea 14)

In [ ]:
# (12) Resumen final
speedup_vs_python = T_PYTHON_SEQ / t_search
print("================= RESUMEN FASE 4 (CUDA) =================")
print(f"GPU                : {GPU_NAME}")
print(f"best_W             : {best_W}")
print(f"best_AUC           : {best_auc:.6f}")
print(f"consistencia       : {consistency:.4f}")
print(f"time_seconds (W*)  : {t_search:.6f} s")
print(f"kernel puro        : {kernel_ms:.3f} ms")
print(f"candidates/s       : {K / t_search:,.0f}")
print(f"speedup / eff      : 1.0 / 1.0  (workers=1, 1 GPU)")
print(f"speedup_vs_python  : {speedup_vs_python:.2f}x  (objetivo >= 5x, RNF-03)")
print("--------------------------------------------------------")
print("Criterios de salida (active-tasks tarea 14):")
print(f"  [{'x' if SELFTEST_TIE_OK else ' '}] self-test empate -> 0.875")
print(f"  [{'x' if SELFTEST_SMALL_OK else ' '}] self-test n=3/K=100 vs mirror SplitMix64 (|dAUC|<1e-4)")
print(f"  [{'x' if abs(best_auc - AUC_PYTHON_SEQ) < 1e-4 else ' '}] |AUC_CUDA - AUC_PySeq| < 1e-4 (n_50/K=100k)")
print(f"  [{'x' if 0.5 <= best_auc <= 1.0 else ' '}] AUC in [0.5,1.0]")
print(f"  [{'x' if consistency >= 0.8 else ' '}] consistencia >= 0.8")
print(f"  [{'x' if speedup_vs_python >= 5 else ' '}] speedup_vs_python >= 5x (GPU: {GPU_NAME})")
print("========================================================")
